# RI #1: Bring your own corpus

This notebook builds a small evaluation set from a folder of your documents and ranks RI #1's
retrieval arms on it. Your corpus has no labeled questions, so the questions are generated: each
one is written from a statement quoted out of a document, and the quote's character offsets are
its gold evidence, which any chunker can be scored against.

Four code cells do the work: point the notebook at a folder, generate a set on your models,
score the arms, and report the spend. With `RUN_EXPENSIVE = False` and no folder named, the
notebook uses ten ContractNLI contracts as a stand-in and reads a reference set built on Vector's
on-prem models, with no model call. Name your folder and set the flag to `True` to build your
own.

The method and its limits are in
`../special_topics/synthetic_question_sets/01_build_and_trust_a_question_set.ipynb`. In short: on
ContractNLI, generated labels ranked these arms close to the human ranking (Kendall's tau 0.86 on
nDCG@10, the normalized discounted cumulative gain at rank 10), their absolute scores ran about 0.14 high, and they favored small chunks. The table
at the end gives an order of arms, and its scores are not comparable with scores on another set.

## 1. Point it at a folder

`FOLDER` is a directory of `.txt`, `.md`, or `.pdf` files, subfolders included. PDFs are
read through their text layer, so a scanned PDF needs optical character recognition (OCR) first, and a document whose answers
sit in tables reads better parsed with Docling, as RI #3 does. `CONFIG_FILE` is a copy of
`aieng-advanced-rag/aieng/advanced_rag/evaluation/synthetic/team-config.example.yaml` with your
domain, your users, and a few real questions as style examples. Without one, the notebook uses a
generic configuration at small sizes.

The generator is your generation model and the judge is your judging model (the shared
`gemini-3.8-flash` for paid families), as `choose_models` returns them. The checker comes from a
different family than the generator, so the round trip does not check a model against its own
habits: `gpt-6-luna` for a Google generator, `gemini-3.5-flash-lite` for any other paid generator, and a different on-prem Qwen model for an on-prem generator. To choose another,
pass `roles=Roles.from_models(MODELS, checker="<model_name>")` to `byoc.generate`.

In [1]:
import byoc
from aieng.advanced_rag import spend
from aieng.advanced_rag.cache.paths import root as cache_root
from aieng.advanced_rag.evaluation.synthetic.replay import write_beir
from aieng.advanced_rag.evaluation.synthetic.validation import chunks_for
from aieng.advanced_rag.models import choose_models


# Off reads the stand-in's reference set. On builds and scores on your models.
RUN_EXPENSIVE = False
# A folder of .txt, .md, or .pdf files. None uses ten ContractNLI contracts.
FOLDER = None
# Your copy of team-config.example.yaml, or None.
CONFIG_FILE = None
# Read at most this many files, so a first run stays small.
MAX_DOCUMENTS = 20

MODELS = choose_models(show=True)
documents, is_demo = byoc.read_corpus(FOLDER, limit=MAX_DOCUMENTS)
config = byoc.config_for(CONFIG_FILE, is_demo)
characters = sum(len(d.text) for d in documents)
print(f"{len(documents)} documents, {characters:,} characters{' (the ContractNLI stand-in)' if is_demo else ''}")
print(f"domain: {config.domain}; scope: {config.scope}; {config.passages_per_document} passages per document")

family google
  generation  gemini-3.5-flash-lite   $0.30 in / $2.50 out per 1M tokens
  judging     gemini-3.8-flash        $0.75 in / $3.75 out per 1M tokens; introductory price
  extraction  gemini-3.5-flash-lite   $0.30 in / $2.50 out per 1M tokens
  embeddings  BAAI/bge-small-en-v1.5  local
key budget unknown (the proxy did not answer)
10 documents, 94,165 characters (the ContractNLI stand-in)
domain: non-disclosure agreements between companies; scope: document; 2 passages per document


## 2. Generate a small set

The build holds out a few documents for null questions, samples passages from the rest, asks the
generator for verbatim statements, and writes one question per statement. Each question then has
to pass a round trip (the checker answers from the quote alone and the judge compares that answer
with the reference), a uniqueness check against the nearest other passages, and deduplication.
Null questions are near-miss rewrites of accepted questions and questions written from the
held-out documents, kept only when the checker finds no answer in the top passages of the
indexed documents.

The label says where the set came from. With the flag off and the stand-in folder, it is the
reference set, followed by what building it on your models would cost. With the flag on, the
estimate prints first, and the build stops if it is over 10% of your key's remaining budget, or
over $1 when your key reports no budget and $300 is assumed. Pass `confirm=True` to run it.

In [2]:
result, generated = byoc.generate(documents, config, MODELS, live=RUN_EXPENSIVE, demo=is_demo)
summary = generated.summary()
funnel = generated.meta["funnel"]
print()
print(
    f"{summary['answerable']} answerable and {summary['null']} null questions; held out: {', '.join(generated.meta['documents']['held_out'])}"
)
print(f"{'kind':<16}{'stage':<18}{'entered':>8}{'passed':>8}")
for row in funnel["stages"]:
    print(f"{row['kind']:<16}{row['stage']:<18}{row['entered']:>8}{row['passed']:>8}")
print()
for q in generated.answerable[:3]:
    print(
        f"{q.question}\n  -> {q.reference}  [{q.evidence[0].doc_id}, characters {q.evidence[0].start} to {q.evidence[0].end}]"
    )

question-set reference artifact, built with Qwen3_8-27B on Vector's cluster at $0
             167 calls, 113,545 prompt + 8,662 completion tokens
             your build: gemini-3.5-flash-lite generates, gpt-6-luna checks, gemini-3.8-flash judges; about 167 calls, est. $0.04 (set RUN_EXPENSIVE = True to build)

27 answerable and 7 null questions; held out: 5.txt, 6.txt
kind            stage              entered  passed
single_hop      quote_located           31      29
single_hop      question_written        29      29
single_hop      round_trip              29      29
single_hop      uniqueness              29      27
single_hop      deduplicated            27      27
near_miss_null  null_written             8       8
near_miss_null  null_check               8       4
held_out_null   quote_located           14      14
held_out_null   null_written             6       5
held_out_null   null_check               5       3
near_miss_null  deduplicated             4       4
held_out_null  

## 3. Score RI #1's arms on the set

The eight arms are the ones the ContractNLI validation compared: BM25, dense
`bge-small-en-v1.5`, hybrid reciprocal rank fusion at dense weights 0.5 and 0.2, dense with a
MiniLM reranker, and a shuffled control at 1,200-character chunks, and BM25 and dense at
600-character chunks. They index the documents the set was built over, minus the held-out ones.
A set built with `scope: corpus` is searched across the whole folder, and `scope: document`
searches each question's own document, as RI #1 does for contracts.

Scoring is local, with no model call. With the flag off it reads cached scores for a set the cache
already holds. With it on it embeds the chunks on the CPU, which takes a few minutes for a small
folder. MRR is mean reciprocal rank. The last column is the top arm's nDCG@10 minus this arm's, with a
95% paired bootstrap interval over the questions. An interval that includes zero means this set
cannot tell the two arms apart.

The cell also writes the set and its 1,200-character chunking in the layout of Benchmarking Information Retrieval (BEIR), so tools that read
BEIR's query relevance judgments, or qrels (`pytrec_eval`, `ranx`, LlamaIndex's `EmbeddingQAFinetuneDataset`), can score your own
retrievers against it.

In [3]:
scores = byoc.score(result, generated, documents, live=RUN_EXPENSIVE)
mean = scores["mean"]
versus = scores["versus_best_ndcg@10"]
print(
    f"{scores['questions']} answerable questions over {scores['documents_indexed']} indexed documents{', corpus-wide' if scores['corpus_wide'] else ', each question in its own document'}"
)
print(f"{'rank':<6}{'arm':<24}{'nDCG@10':>9}{'recall@3':>10}{'MRR':>7}   nDCG@10 against {versus['best']}")
order = sorted(mean["ndcg@10"], key=lambda arm: -mean["ndcg@10"][arm])
for rank, arm in enumerate(order, 1):
    gap = versus["arms"].get(arm)
    interval = f"{gap['mean_difference']:+.3f} [{gap['low']:+.3f}, {gap['high']:+.3f}]" if gap else "top arm"
    print(
        f"{rank:<6}{arm:<24}{mean['ndcg@10'][arm]:>9.3f}{mean['recall@3'][arm]:>10.3f}{mean['mrr'][arm]:>7.3f}   {interval}"
    )

indexed = set(generated.meta["documents"]["indexed"])
beir_dir = cache_root() / "byoc-demo" / "beir" / result.digest[:12]
written = write_beir(
    generated.variant("plain"), chunks_for([d for d in documents if d.doc_id in indexed], 1200, 200), beir_dir
)
print(
    f"\nBEIR export: {written['queries']} queries, {written['corpus']} chunks, {written['qrels']} qrels rows, under the local cache"
)

arm scores: cached, for the set 6204e405bea4
27 answerable questions over 8 indexed documents, each question in its own document
rank  arm                       nDCG@10  recall@3    MRR   nDCG@10 against dense+rerank@1200
1     dense+rerank@1200           0.938     0.920  0.957   top arm
2     hybrid-rrf-0.2@1200         0.897     0.883  0.886   +0.040 [-0.025, +0.110]
3     hybrid-rrf-0.5@1200         0.896     0.883  0.872   +0.041 [-0.030, +0.113]
4     bm25@1200                   0.887     0.846  0.864   +0.050 [-0.012, +0.119]
5     bm25@600                    0.862     0.840  0.890   +0.076 [-0.002, +0.147]
6     dense@600                   0.826     0.704  0.902   +0.112 [+0.039, +0.190]
7     dense@1200                  0.817     0.827  0.815   +0.120 [+0.051, +0.203]
8     shuffled-control@1200       0.379     0.235  0.270   +0.559 [+0.469, +0.645]

BEIR export: 34 queries, 83 chunks, 43 qrels rows, under the local cache


On the stand-in contracts, the reranker arm comes first, and the two hybrids and both BM25 arms
sit within an interval of it that includes zero, so 27 questions cannot separate those five. Both
dense arms and the shuffled control are separated from it. On ContractNLI, generated labels also
put the reranker first and the control last, and gave 21 of the 22 arm pairs that human labels
separated the same sign.

Three cautions from the ContractNLI validation carry over to your corpus. Generated labels score
every arm higher than human questions would. They rate 600-character dense retrieval about level
with 1,200-character dense, where human labels put it 0.10 nDCG@10 lower. A set generated this
way says nothing about which answer model is better.

To grow the set, raise `MAX_DOCUMENTS` or `passages_per_document` in your configuration file.
The cost line in section 2 scales with both. Before you rely on the ranking, read 20 to 30
questions with someone who knows the documents and count the correct reference answers.

In [4]:
print(spend.report())

this run, estimated   est. $0.00 over 0 models
